In [12]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras

In [13]:
np.random.seed(42)
tf.random.set_seed(42)

# Chuẩn bị dữ liệu


Ta sử dụng dataset **Emotions dataset for NLP** ([Nguồn](https://www.kaggle.com/datasets/praveengovi/emotions-dataset-for-nlp)).
Dữ liệu đã được xử lý trước để thuận tiện cho việc xây mô hình.


In [37]:
df = pd.read_csv("../data/NLP.csv")
df.head()

,statement,emotion
0,im feeling rather rotten so im not very ambiti...,sadness
1,im updating my blog because i feel shitty,sadness
2,i never make her separate from me because i do...,sadness
3,i left with my bouquet of red and yellow tulip...,joy
4,i was feeling a little vain when i did this one,sadness


Để xây mô hình, ta cần gán các cảm xúc cần dự đoán cho một chỉ số `emotion id` như dưới đây:


In [51]:
emotion_map = {
    "surprise": 0,
    "joy": 1,
    "sadness": 2,
    "anger": 3,
    "fear": 4,
    "love": 5,
}

df['emotion id'] = df['emotion'].map(emotion_map)
df.head()

,statement,emotion,emotion id
0,i left feeling disappointed in her knowledge,sadness,2
1,i usually feel regretful and guilty after the ...,sadness,2
2,i am happy to be feeling well enough to be bac...,joy,1
3,i get the feeling that she is dissatisfied wit...,anger,3
4,im feeling quite positive at the moment,joy,1


Ta xáo trộn dữ liệu trước khi tách để giúp tách tập dữ liệu cân đối hơn.


In [52]:
df = df.sample(frac=1).reset_index(drop=True)
df.head(10)

,statement,emotion,emotion id
0,im feeling discouraged sad angry afraid of tom...,sadness,2
1,i also has the meaning of trusting oneself tru...,sadness,2
2,i feel like i m worthless and i can t do any g...,sadness,2
3,i can feel something inside me something delic...,love,5
4,i feel like i know i m troubled and that s why...,sadness,2
5,i feel as i did when i was troubled easily agi...,sadness,2
6,when my grandmother came to stay with us perma...,anger,3
7,i feel that he wasn t making the effort to see...,sadness,2
8,i embraced feeling thankful that the middle wa...,joy,1
9,i get another call from a frantic junior for m...,anger,3


Tiến hành chia tập dữ liệu:

- Dữ liệu chia thành hai tập `raw` và `test` với tỉ lệ 8:2,
- Sau đó, chia tập `raw` thành `train` và `val` với tỉ lệ 7:1.

Như vậy, tỉ lệ chia dữ liệu là `train`:`val`:`test` = 7:1:2.


In [53]:
from sklearn.model_selection import train_test_split

X = df['statement'].values
y = df['emotion id'].values

X_raw, X_test, y_raw, y_test = train_test_split(X,y,
                                                   test_size = 0.2,
                                                   random_state = 42)
X_raw = np.array(X_raw)
X_test = np.array(X_test)
y_raw = np.array(y_raw)
y_test = np.array(y_test)

X_train, X_val, y_train, y_val = train_test_split(X_raw, y_raw,
                                                 test_size=0.125,
                                                 random_state=42)

X_train = np.array(X_train)
X_val = np.array(X_val)
y_train = np.array(y_train)
y_val = np.array(y_val)

X_train.shape, X_val.shape, X_test.shape

((14000,), (2000,), (4000,))

# Xây dựng mô hình


Ta sử dụng tokenizer trước để xử lý các từ/cụm từ/kí tự đặc biệt thành số mà máy tính có thể hiểu được. Đây là quá trình **tokenize**


In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import pad_sequences

vocab_size = 10000
max_length = 100

tokenizer = Tokenizer(num_words = vocab_size, oov_token = "OOV") #out of vocabulary
tokenizer.fit_on_texts(X_train)

X_train_seqs = tokenizer.texts_to_sequences(X_train)
X_train_padded = pad_sequences(X_train_seqs, maxlen=max_length, padding="post")

X_val_seqs = tokenizer.texts_to_sequences(X_val)
X_val_padded = pad_sequences(X_val_seqs, maxlen=max_length, padding="post")

len(X_train_padded),len(X_val_padded)

(14000, 2000)

Bây giờ, ta sẽ tiến hành xây mô hình NLP.


In [ ]:
num_classes = len(emotion_map)

embedding_dim = 128

model = keras.Sequential([
    keras.Input(shape=(max_length,)),
    keras.layers.Embedding(vocab_size, embedding_dim, trainable=True),
    keras.layers.Flatten(),
    keras.layers.BatchNormalization(),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(num_classes, activation='softmax'),
])

model.compile(optimizer='adam',
             loss='sparse_categorical_crossentropy',
             metrics=['accuracy']
)
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (None, 150, 64)        │       320,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 9600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 9600)           │        38,400 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 48)             │       460,848 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 48)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 6)              │           294 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 819,542 (3.13 MB)

 Trainable params: 800,342 (3.05 MB)

 Non-trainable params: 19,200 (75.00 KB)

# Huấn luyện mô hình


Ta bắt đầu huấn luyện mô hình với dữ liệu train và val đã xử lý ở trên.


In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint

checkpoint = ModelCheckpoint(
    filepath = "NLP_ep{epoch:02d}.h5",
    save_weights_only = False,
    save_best_only = True,
    monitor = "val_loss",
    verbose = 1
)

history = model.fit(
    X_train_padded, y_train,
    validation_data = (X_val_padded, y_val),
    epochs = 15,
    callbacks = [checkpoint],
)

Epoch 1/15
432/438 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.3182 - loss: 1.6478
Epoch 1: saving model to NLP_ep01.h5


438/438 ━━━━━━━━━━━━━━━━━━━━ 12s 13ms/step - accuracy: 0.3559 - loss: 1.5532 - val_accuracy: 0.4710 - val_loss: 1.3438
Epoch 2/15
436/438 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.6431 - loss: 0.9480
Epoch 2: saving model to NLP_ep02.h5


438/438 ━━━━━━━━━━━━━━━━━━━━ 20s 45ms/step - accuracy: 0.6910 - loss: 0.8273 - val_accuracy: 0.7975 - val_loss: 0.6039
Epoch 3/15
433/438 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.8623 - loss: 0.4020
Epoch 3: saving model to NLP_ep03.h5


438/438 ━━━━━━━━━━━━━━━━━━━━ 6s 13ms/step - accuracy: 0.8721 - loss: 0.3714 - val_accuracy: 0.8230 - val_loss: 0.5569
Epoch 4/15
437/438 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9249 - loss: 0.2250
Epoch 4: saving model to NLP_ep04.h5


438/438 ━━━━━━━━━━━━━━━━━━━━ 6s 14ms/step - accuracy: 0.9309 - loss: 0.2093 - val_accuracy: 0.8290 - val_loss: 0.6521
Epoch 5/15
433/438 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9527 - loss: 0.1525
Epoch 5: saving model to NLP_ep05.h5


...

438/438 ━━━━━━━━━━━━━━━━━━━━ 6s 13ms/step - accuracy: 0.9752 - loss: 0.0737 - val_accuracy: 0.8165 - val_loss: 1.0680
Epoch 12/15
434/438 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9811 - loss: 0.0583
Epoch 12: saving model to NLP_ep12.h5


438/438 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.9797 - loss: 0.0638 - val_accuracy: 0.8290 - val_loss: 1.0889
Epoch 13/15
436/438 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.9792 - loss: 0.0615
Epoch 13: saving model to NLP_ep13.h5


438/438 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.9803 - loss: 0.0612 - val_accuracy: 0.8255 - val_loss: 1.0709
Epoch 14/15
435/438 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9785 - loss: 0.0634
Epoch 14: saving model to NLP_ep14.h5


438/438 ━━━━━━━━━━━━━━━━━━━━ 10s 11ms/step - accuracy: 0.9782 - loss: 0.0666 - val_accuracy: 0.8300 - val_loss: 1.1256
Epoch 15/15
434/438 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.9773 - loss: 0.0721
Epoch 15: saving model to NLP_ep15.h5


438/438 ━━━━━━━━━━━━━━━━━━━━ 6s 13ms/step - accuracy: 0.9767 - loss: 0.0750 - val_accuracy: 0.8225 - val_loss: 1.1639


# Kiểm tra mô hình


Bây giờ, ta kiểm tra độ chính xác của mô hình bằng cách kiểm tra.


In [57]:
X_test_seqs = tokenizer.texts_to_sequences(X_test)
X_test_padded = pad_sequences(X_test_seqs, padding='post', truncating='post', maxlen=max_length)

In [58]:
from tensorflow.keras.models import load_model

model = load_model("NLP_ep15.h5")
preds = model.predict(X_test_padded)

125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step


# Đánh giá mô hình


Ta tiến hành đánh giá mô hình trên bằng các chỉ số:

- `accuracy`: độ chính xác của mô hình.
- `f1-score`: chỉ số đánh giá mô hình bằng cách kết hợp Precision (độ chính xác) và Recall (độ bao phủ). Nó được tính bằng trung bình điều hòa của hai giá trị này:


In [64]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

pred_classes = np.argmax(preds, axis=1)

accuracy = accuracy_score(y_test, pred_classes)
print(f"Test Accuracy: {accuracy * 100:.2f}%\n")

f1 = f1_score(y_test, pred_classes, average='weighted')
print(f"Weighted F1-Score: {f1 * 100:.2f}%")


Test Accuracy: 82.12%

Weighted F1-Score: 82.10%
